# 03. Leakage-Safe Preprocessing

This notebook turns the current AquaSense AI pipeline artifacts into a modelling matrix for breach prediction and pollutant forecasting.

Design constraints:

- use the current files on disk, not assumed schemas;
- sort and regularise by `facility_id` and `timestamp`;
- use only current/past values for predictors;
- exclude post-hoc incident labels, future labels, current compliance explanations, lab result columns, and recommendations from predictors;
- recompute future labels with unknown tail rows left missing before supervised filtering;
- split temporally before fitting imputers or feature filters;
- write auditable artifacts for the training notebook.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
CONFIG = ROOT / "packages" / "shared" / "config"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

OUTPUTS = {
    "model_matrix": PROCESSED / "preprocessed_model_matrix.csv",
    "feature_manifest": PROCESSED / "preprocessing_feature_manifest.csv",
    "split_summary": PROCESSED / "temporal_split_summary.csv",
    "train_medians": PROCESSED / "preprocessing_train_medians.csv",
    "audit": PROCESSED / "preprocessing_audit.json",
}
OUTPUTS


## 2. Load Current Pipeline Artifacts

The operational dataset is the preprocessing source of truth because it joins direct sensors, soft-sensor estimates, lab samples, compliance labels, and future labels. Raw files are loaded for shape and time-range cross-checks.


In [ ]:
required_paths = {
    "direct sensors": RAW / "direct_sensor_readings.csv",
    "lab results": RAW / "lab_results.csv",
    "soft sensor estimates": PROCESSED / "soft_sensor_estimates.csv",
    "operational compliance": PROCESSED / "operational_compliance_dataset.csv",
    "demo limits": CONFIG / "demo_limits.json",
}

missing_paths = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(f"Missing required pipeline artifacts: {missing_paths}")

sensors = pd.read_csv(required_paths["direct sensors"], parse_dates=["timestamp"])
lab = pd.read_csv(required_paths["lab results"], parse_dates=["timestamp"])
estimates = pd.read_csv(required_paths["soft sensor estimates"], parse_dates=["timestamp"])
operational = pd.read_csv(required_paths["operational compliance"], parse_dates=["timestamp"], low_memory=False)

with open(required_paths["demo limits"], encoding="utf-8") as f:
    consent_limits = json.load(f)

artifact_shapes = pd.DataFrame([
    ("direct sensors", *sensors.shape, sensors["timestamp"].min(), sensors["timestamp"].max()),
    ("lab results", *lab.shape, lab["timestamp"].min(), lab["timestamp"].max()),
    ("soft sensor estimates", *estimates.shape, estimates["timestamp"].min(), estimates["timestamp"].max()),
    ("operational compliance", *operational.shape, operational["timestamp"].min(), operational["timestamp"].max()),
], columns=["artifact", "rows", "columns", "start", "end"])
artifact_shapes


## 3. Schema and Availability Contract


In [ ]:
required_columns = {
    "timestamp", "facility_id", "breach_now",
    "flow_rate_lps", "temperature_c", "ph", "turbidity_ntu",
    "conductivity_us_cm", "dissolved_oxygen_mg_l", "orp_mv",
    "ammonia_mg_l", "uv254_abs", "sensor_status",
    "estimated_tss_mg_l", "estimated_cod_mg_l", "estimated_bod_mg_l",
    "tss_mg_l_for_compliance", "cod_mg_l_for_compliance",
    "bod_mg_l_for_compliance", "ammonia_mg_l_for_compliance",
}
missing_columns = sorted(required_columns.difference(operational.columns))
if missing_columns:
    raise ValueError(f"Operational dataset is missing required columns: {missing_columns}")

post_hoc_or_unavailable = sorted({
    "event_type",
    "lab_tss_mg_l", "lab_cod_mg_l", "lab_bod_mg_l", "lab_ammonia_mg_l",
    "lab_result_available",
    "quality_value_source", "tss_value_source", "cod_value_source", "bod_value_source", "ammonia_value_source",
    "compliance_status", "breached_parameters", "current_breach_reason",
    "main_risk_driver", "recommended_action", "margin_to_limit_pct",
    "breach_now", "breach_next_15min", "breach_next_30min", "breach_next_60min",
}.intersection(operational.columns))

availability_contract = pd.DataFrame({"column": sorted(operational.columns)})
availability_contract["excluded_from_predictors"] = availability_contract["column"].isin(post_hoc_or_unavailable)
availability_contract["reason"] = np.select(
    [
        availability_contract["column"].str.startswith("breach_next_"),
        availability_contract["column"].eq("breach_now"),
        availability_contract["column"].str.startswith("lab_"),
        availability_contract["column"].eq("event_type"),
        availability_contract["column"].isin(["compliance_status", "breached_parameters", "current_breach_reason", "main_risk_driver", "recommended_action"]),
        availability_contract["column"].str.endswith("_value_source") | availability_contract["column"].eq("quality_value_source"),
        availability_contract["column"].isin(["timestamp", "facility_id", "sensor_status"]),
    ],
    [
        "future target label",
        "current outcome label, retained only for target construction/audit",
        "sparse lab result; not assumed available in real time",
        "post-hoc incident label from simulation, not real-time telemetry",
        "derived compliance explanation or operator guidance",
        "source metadata can reveal lab availability/process provenance",
        "metadata or categorical handled separately",
    ],
    default="candidate real-time numeric input",
)
availability_contract


## 4. Sort, Deduplicate, and Regularise Time Grid


In [ ]:
sample_minutes = int(consent_limits.get("sample_interval_minutes", 5))
sample_freq = f"{sample_minutes}min"

work = operational.copy()
work["timestamp"] = pd.to_datetime(work["timestamp"], errors="coerce")
if work["timestamp"].isna().any():
    raise ValueError("Unparseable timestamps found in operational dataset")

before_dedup = len(work)
work = (
    work.sort_values(["facility_id", "timestamp"])
    .drop_duplicates(["facility_id", "timestamp"], keep="last")
    .reset_index(drop=True)
)
duplicate_rows_removed = before_dedup - len(work)

regularised_parts = []
inserted_timestamp_rows = 0
for facility_id, group in work.groupby("facility_id", sort=False):
    group = group.sort_values("timestamp").set_index("timestamp")
    full_index = pd.date_range(group.index.min(), group.index.max(), freq=sample_freq)
    inserted_timestamp_rows += int(len(full_index) - len(group.index))
    regular = group.reindex(full_index)
    regular.index.name = "timestamp"
    regular["facility_id"] = facility_id
    regular["was_inserted_timestamp"] = regular.drop(columns=["facility_id"], errors="ignore").isna().all(axis=1)
    regularised_parts.append(regular.reset_index())

work = pd.concat(regularised_parts, ignore_index=True).sort_values(["facility_id", "timestamp"]).reset_index(drop=True)

cadence_audit = []
for facility_id, group in work.groupby("facility_id", sort=False):
    diffs = group["timestamp"].diff().dropna().dt.total_seconds().div(60)
    cadence_audit.append({
        "facility_id": facility_id,
        "rows": len(group),
        "start": group["timestamp"].min(),
        "end": group["timestamp"].max(),
        "duplicate_rows_removed": duplicate_rows_removed,
        "inserted_timestamp_rows": inserted_timestamp_rows,
        "most_common_interval_minutes": diffs.mode().iloc[0] if len(diffs) else np.nan,
        "unique_intervals": int(diffs.nunique()) if len(diffs) else 0,
    })
pd.DataFrame(cadence_audit)


## 5. Recompute Future Targets Without Safe-Tail Leakage


In [ ]:
def future_any_target(series: pd.Series, steps: int) -> pd.Series:
    numeric = series.astype("float")
    future = pd.concat([numeric.shift(-i) for i in range(1, steps + 1)], axis=1)
    has_unknown_future = future.isna().any(axis=1)
    target = future.max(axis=1).astype("float")
    target[has_unknown_future] = np.nan
    return target.astype("Float64")

horizons = list(consent_limits.get("prediction_horizons_minutes", [15, 30, 60]))
max_horizon = max(horizons)
max_horizon_steps = max_horizon // sample_minutes

work["breach_now_bool"] = work["breach_now"].astype("boolean")

for minutes in horizons:
    steps = minutes // sample_minutes
    target_col = f"target_breach_next_{minutes}min"
    work[target_col] = (
        work.groupby("facility_id", group_keys=False)["breach_now_bool"]
        .apply(lambda s: future_any_target(s, steps))
        .reset_index(level=0, drop=True)
    )

forecast_targets = {
    "target_cod_mg_l_plus_30min": "cod_mg_l_for_compliance",
    "target_tss_mg_l_plus_30min": "tss_mg_l_for_compliance",
    "target_bod_mg_l_plus_30min": "bod_mg_l_for_compliance",
    "target_ammonia_mg_l_plus_30min": "ammonia_mg_l_for_compliance",
    "target_ph_plus_30min": "ph",
}
forecast_steps = 30 // sample_minutes
for target_col, source_col in forecast_targets.items():
    work[target_col] = work.groupby("facility_id", group_keys=False)[source_col].shift(-forecast_steps)

target_cols = [f"target_breach_next_{m}min" for m in horizons]
forecast_target_cols = list(forecast_targets)

pd.DataFrame({
    "target": target_cols + forecast_target_cols,
    "missing_rows": [int(work[col].isna().sum()) for col in target_cols + forecast_target_cols],
    "positive_rate_pct": [round(work[col].dropna().mean() * 100, 3) if col in target_cols else np.nan for col in target_cols + forecast_target_cols],
})


## 6. Physical Validity Flags and Causal Cleaning


In [ ]:
realtime_inputs = [
    "flow_rate_lps", "temperature_c", "ph", "turbidity_ntu",
    "conductivity_us_cm", "dissolved_oxygen_mg_l", "orp_mv",
    "ammonia_mg_l", "uv254_abs",
    "estimated_tss_mg_l", "estimated_cod_mg_l", "estimated_bod_mg_l",
]

physical_ranges = {
    "flow_rate_lps": (0, None),
    "temperature_c": (0, 80),
    "ph": (0, 14),
    "turbidity_ntu": (0, None),
    "conductivity_us_cm": (0, None),
    "dissolved_oxygen_mg_l": (0, 20),
    "orp_mv": (-1000, 1000),
    "ammonia_mg_l": (0, None),
    "uv254_abs": (0, None),
    "estimated_tss_mg_l": (0, None),
    "estimated_cod_mg_l": (0, None),
    "estimated_bod_mg_l": (0, None),
}

clean_cols = []
validity_rows = []
short_gap_limit_steps = 15 // sample_minutes

for col in realtime_inputs:
    lower, upper = physical_ranges[col]
    raw = pd.to_numeric(work[col], errors="coerce")
    valid = raw.notna()
    if lower is not None:
        valid &= raw >= lower
    if upper is not None:
        valid &= raw <= upper

    clean_col = f"{col}_clean"
    flag_col = f"{col}_missing_or_invalid"
    work[flag_col] = (~valid).astype(int)
    work[clean_col] = raw.where(valid)
    work[clean_col] = work.groupby("facility_id", group_keys=False)[clean_col].ffill(limit=short_gap_limit_steps)
    clean_cols.append(clean_col)

    validity_rows.append({
        "source_column": col,
        "clean_column": clean_col,
        "missing_or_invalid_before_cleaning": int((~valid).sum()),
        "missing_after_short_ffill": int(work[clean_col].isna().sum()),
        "physical_lower": lower,
        "physical_upper": upper,
    })

validity_audit = pd.DataFrame(validity_rows)
validity_audit


## 7. Real-Time Aliases and Data-Quality Features


In [ ]:
alias_map = {
    "flow_rate_lps_clean": "flow_rate_lps_rt",
    "temperature_c_clean": "temperature_c_rt",
    "ph_clean": "ph_rt",
    "turbidity_ntu_clean": "turbidity_ntu_rt",
    "conductivity_us_cm_clean": "conductivity_us_cm_rt",
    "dissolved_oxygen_mg_l_clean": "dissolved_oxygen_mg_l_rt",
    "orp_mv_clean": "orp_mv_rt",
    "ammonia_mg_l_clean": "ammonia_mg_l_rt",
    "uv254_abs_clean": "uv254_abs_rt",
    "estimated_tss_mg_l_clean": "tss_mg_l_rt",
    "estimated_cod_mg_l_clean": "cod_mg_l_rt",
    "estimated_bod_mg_l_clean": "bod_mg_l_rt",
}
for source_col, alias_col in alias_map.items():
    work[alias_col] = work[source_col]

rt_cols = list(alias_map.values())
missing_flag_cols = [f"{col}_missing_or_invalid" for col in realtime_inputs]
work["sensor_status_ok"] = work["sensor_status"].fillna("missing").eq("ok").astype(int)
work["sensor_status_missing_burst"] = work["sensor_status"].fillna("missing").eq("missing_burst").astype(int)
work["missing_or_invalid_sensor_count"] = work[missing_flag_cols].sum(axis=1)
work["was_inserted_timestamp"] = work["was_inserted_timestamp"].fillna(False).astype(int)

missing_count_series = work["missing_or_invalid_sensor_count"] + work["was_inserted_timestamp"]
work["missing_or_invalid_count_60min"] = (
    missing_count_series.groupby(work["facility_id"])
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

jump_thresholds = {
    "ph_rt": 1.5, "temperature_c_rt": 5.0, "flow_rate_lps_rt": 40.0,
    "turbidity_ntu_rt": 150.0, "conductivity_us_cm_rt": 450.0,
    "dissolved_oxygen_mg_l_rt": 3.0, "orp_mv_rt": 150.0,
    "ammonia_mg_l_rt": 50.0, "uv254_abs_rt": 0.8,
    "tss_mg_l_rt": 350.0, "cod_mg_l_rt": 500.0, "bod_mg_l_rt": 300.0,
}
for col, threshold in jump_thresholds.items():
    diff = work.groupby("facility_id")[col].diff().abs()
    work[f"{col}_jump_flag"] = diff.gt(threshold).fillna(False).astype(int)

work["any_sensor_jump_flag"] = work[[f"{col}_jump_flag" for col in jump_thresholds]].max(axis=1)
quality_feature_cols = [
    "sensor_status_ok", "sensor_status_missing_burst", "was_inserted_timestamp",
    "missing_or_invalid_sensor_count", "missing_or_invalid_count_60min", "any_sensor_jump_flag",
]
work[["timestamp", "facility_id"] + quality_feature_cols].head()


## 8. Causal Lag, Rolling, and Trend Features


In [ ]:
def rolling_slope(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(values)
    if mask.sum() < 3:
        return np.nan
    x = np.arange(len(values), dtype=float)[mask]
    y = values[mask]
    return float(np.polyfit(x, y, 1)[0])

feature_cols = []
feature_cols.extend(rt_cols)
feature_cols.extend(quality_feature_cols)
feature_cols.extend(missing_flag_cols)

lag_steps = {1: "5min", 3: "15min", 6: "30min", 12: "60min", 24: "120min"}
rolling_windows = {3: "15min", 6: "30min", 12: "60min", 24: "120min"}

for col in rt_cols:
    grouped = work.groupby("facility_id", group_keys=False)[col]

    for steps, label in lag_steps.items():
        new_col = f"{col}_lag_{label}"
        work[new_col] = grouped.shift(steps)
        feature_cols.append(new_col)

    for steps, label in rolling_windows.items():
        roll = grouped.rolling(window=steps, min_periods=max(2, steps // 2))
        mean_col = f"{col}_rolling_{label}_mean"
        std_col = f"{col}_rolling_{label}_std"
        min_col = f"{col}_rolling_{label}_min"
        max_col = f"{col}_rolling_{label}_max"
        median_col = f"{col}_rolling_{label}_median"
        range_col = f"{col}_rolling_{label}_range"
        work[mean_col] = roll.mean().reset_index(level=0, drop=True)
        work[std_col] = roll.std().reset_index(level=0, drop=True)
        work[min_col] = roll.min().reset_index(level=0, drop=True)
        work[max_col] = roll.max().reset_index(level=0, drop=True)
        work[median_col] = roll.median().reset_index(level=0, drop=True)
        work[range_col] = work[max_col] - work[min_col]
        feature_cols.extend([mean_col, std_col, min_col, max_col, median_col, range_col])

    for steps, label in {1: "5min", 3: "15min", 6: "30min"}.items():
        delta_col = f"{col}_delta_{label}"
        work[delta_col] = work[col] - grouped.shift(steps)
        feature_cols.append(delta_col)

    pct_col = f"{col}_pct_change_30min"
    denom = grouped.shift(6).replace(0, np.nan)
    work[pct_col] = ((work[col] - denom) / denom).replace([np.inf, -np.inf], np.nan)
    feature_cols.append(pct_col)

    slope_col = f"{col}_rolling_30min_slope"
    work[slope_col] = grouped.rolling(window=6, min_periods=4).apply(rolling_slope, raw=True).reset_index(level=0, drop=True)
    feature_cols.append(slope_col)

len(feature_cols), feature_cols[:20]


## 9. Compliance Margin, Load, Ratio, and Time Features


In [ ]:
L = consent_limits["limits"]

def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

margin_features = {
    "ph_lower_margin_rt": work["ph_rt"] - L["ph_min"],
    "ph_upper_margin_rt": L["ph_max"] - work["ph_rt"],
    "temperature_margin_rt": L["temperature_max_c"] - work["temperature_c_rt"],
    "cod_margin_rt": L["cod_max_mg_l"] - work["cod_mg_l_rt"],
    "bod_margin_rt": L["bod_max_mg_l"] - work["bod_mg_l_rt"],
    "tss_margin_rt": L["tss_max_mg_l"] - work["tss_mg_l_rt"],
    "ammonia_margin_rt": L["ammonia_max_mg_l"] - work["ammonia_mg_l_rt"],
}
for col, values in margin_features.items():
    work[col] = values

margin_pct_cols = []
margin_denominators = {
    "ph_lower_margin_rt": L["ph_min"],
    "ph_upper_margin_rt": L["ph_max"],
    "temperature_margin_rt": L["temperature_max_c"],
    "cod_margin_rt": L["cod_max_mg_l"],
    "bod_margin_rt": L["bod_max_mg_l"],
    "tss_margin_rt": L["tss_max_mg_l"],
    "ammonia_margin_rt": L["ammonia_max_mg_l"],
}
for col, denom in margin_denominators.items():
    pct_col = col.replace("_rt", "_pct_rt")
    work[pct_col] = work[col] / denom * 100
    margin_pct_cols.append(pct_col)

work["min_margin_pct_rt"] = work[margin_pct_cols].min(axis=1)
feature_cols.extend(list(margin_features) + margin_pct_cols + ["min_margin_pct_rt"])

load_features = {
    "cod_load_rt": work["cod_mg_l_rt"] * work["flow_rate_lps_rt"],
    "bod_load_rt": work["bod_mg_l_rt"] * work["flow_rate_lps_rt"],
    "tss_load_rt": work["tss_mg_l_rt"] * work["flow_rate_lps_rt"],
    "ammonia_load_rt": work["ammonia_mg_l_rt"] * work["flow_rate_lps_rt"],
}
for col, values in load_features.items():
    work[col] = values
feature_cols.extend(load_features)

ratio_features = {
    "bod_cod_ratio_rt": safe_divide(work["bod_mg_l_rt"], work["cod_mg_l_rt"]),
    "tss_turbidity_ratio_rt": safe_divide(work["tss_mg_l_rt"], work["turbidity_ntu_rt"]),
    "cod_uv254_ratio_rt": safe_divide(work["cod_mg_l_rt"], work["uv254_abs_rt"]),
    "do_cod_ratio_rt": safe_divide(work["dissolved_oxygen_mg_l_rt"], work["cod_mg_l_rt"]),
}
for col, values in ratio_features.items():
    work[col] = values.replace([np.inf, -np.inf], np.nan)
feature_cols.extend(ratio_features)

work["hour"] = work["timestamp"].dt.hour
work["day_of_week"] = work["timestamp"].dt.dayofweek
work["is_weekend"] = work["day_of_week"].isin([5, 6]).astype(int)
work["is_working_hour"] = work["hour"].between(7, 18).astype(int)
work["sin_hour"] = np.sin(2 * np.pi * work["hour"] / 24)
work["cos_hour"] = np.cos(2 * np.pi * work["hour"] / 24)
work["sin_day_of_week"] = np.sin(2 * np.pi * work["day_of_week"] / 7)
work["cos_day_of_week"] = np.cos(2 * np.pi * work["day_of_week"] / 7)
time_feature_cols = ["hour", "day_of_week", "is_weekend", "is_working_hour", "sin_hour", "cos_hour", "sin_day_of_week", "cos_day_of_week"]
feature_cols.extend(time_feature_cols)
len(feature_cols)


## 10. Temporal Split Before Fitting Anything


In [ ]:
max_lag_steps = max(lag_steps)
work["row_number_by_facility"] = work.groupby("facility_id").cumcount()
work["has_full_history_window"] = work["row_number_by_facility"].ge(max_lag_steps)

supervised_mask = work["has_full_history_window"]
for col in target_cols:
    supervised_mask &= work[col].notna()

supervised = work.loc[supervised_mask].copy().reset_index(drop=True)
supervised = supervised.sort_values(["timestamp", "facility_id"]).reset_index(drop=True)

n = len(supervised)
train_end = int(np.floor(n * 0.70))
validation_end = int(np.floor(n * 0.85))

supervised["split"] = "test"
supervised.loc[:train_end - 1, "split"] = "train"
supervised.loc[train_end:validation_end - 1, "split"] = "validation"

split_summary = (
    supervised.groupby("split", sort=False)
    .agg(
        rows=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        breach_next_30min_rate=("target_breach_next_30min", "mean"),
        current_breach_rate=("breach_now_bool", "mean"),
    )
    .reset_index()
)
split_summary["breach_next_30min_rate"] = (split_summary["breach_next_30min_rate"] * 100).round(3)
split_summary["current_breach_rate"] = (split_summary["current_breach_rate"] * 100).round(3)
split_summary


## 11. Train-Only Imputation and Feature Filtering


In [ ]:
feature_cols = list(dict.fromkeys(feature_cols))
for col in feature_cols:
    supervised[col] = pd.to_numeric(supervised[col], errors="coerce")

train_mask = supervised["split"].eq("train")
X_train_raw = supervised.loc[train_mask, feature_cols]

all_nan_features = X_train_raw.columns[X_train_raw.isna().all()].tolist()
candidate_feature_cols = [col for col in feature_cols if col not in all_nan_features]

imputer = SimpleImputer(strategy="median")
imputer.fit(supervised.loc[train_mask, candidate_feature_cols])
imputed_values = imputer.transform(supervised[candidate_feature_cols])
imputed_feature_cols = [f"{col}__imputed" for col in candidate_feature_cols]
imputed_df = pd.DataFrame(imputed_values, columns=imputed_feature_cols, index=supervised.index)

train_medians = pd.DataFrame({"feature": candidate_feature_cols, "train_median": imputer.statistics_})
X_train_imputed = imputed_df.loc[train_mask, imputed_feature_cols]

low_variance_features = X_train_imputed.columns[X_train_imputed.nunique(dropna=False).le(1)].tolist()
correlation_threshold = 0.995
remaining_after_variance = [col for col in imputed_feature_cols if col not in low_variance_features]
corr = X_train_imputed[remaining_after_variance].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_drop = sorted({column for column in upper.columns if (upper[column] > correlation_threshold).any()})

kept_imputed_features = [
    col for col in imputed_feature_cols
    if col not in low_variance_features and col not in high_corr_drop
]

manifest = pd.DataFrame({
    "source_feature": candidate_feature_cols,
    "imputed_feature": imputed_feature_cols,
})
manifest["missing_pct_all_rows"] = [round(float(supervised[col].isna().mean() * 100), 4) for col in candidate_feature_cols]
manifest["missing_pct_train"] = [round(float(supervised.loc[train_mask, col].isna().mean() * 100), 4) for col in candidate_feature_cols]
manifest["train_nunique_after_impute"] = [int(X_train_imputed[f"{col}__imputed"].nunique(dropna=False)) for col in candidate_feature_cols]
manifest["excluded_reason"] = ""
manifest.loc[manifest["source_feature"].isin(all_nan_features), "excluded_reason"] = "all NaN in train"
manifest.loc[manifest["imputed_feature"].isin(low_variance_features), "excluded_reason"] = "low variance in train"
manifest.loc[manifest["imputed_feature"].isin(high_corr_drop), "excluded_reason"] = f"correlation > {correlation_threshold} on train"
manifest["keep_for_model"] = manifest["imputed_feature"].isin(kept_imputed_features)

{
    "raw_feature_count": len(feature_cols),
    "candidate_feature_count": len(candidate_feature_cols),
    "all_nan_features": len(all_nan_features),
    "low_variance_features": len(low_variance_features),
    "high_correlation_drops": len(high_corr_drop),
    "kept_features": len(kept_imputed_features),
}


## 12. Leakage Audit


In [ ]:
forbidden_exact = {
    "event_type", "breach_now", "breach_now_bool", "compliance_status",
    "breached_parameters", "current_breach_reason", "main_risk_driver", "recommended_action",
    "lab_result_available", "quality_value_source", "tss_value_source", "cod_value_source",
    "bod_value_source", "ammonia_value_source",
}
forbidden_prefixes = ("target_", "breach_next_", "lab_")
forbidden_substrings = ("future", "plus_", "recommended", "breach_next")

feature_leakage_hits = []
for feature in kept_imputed_features:
    source = feature.replace("__imputed", "")
    if source in forbidden_exact or source.startswith(forbidden_prefixes) or any(token in source for token in forbidden_substrings):
        feature_leakage_hits.append(source)

if feature_leakage_hits:
    raise AssertionError(f"Forbidden leakage-like predictors found: {feature_leakage_hits}")

pd.DataFrame({
    "check": ["forbidden feature names", "target columns missing only at tails", "temporal split order"],
    "status": [
        "pass",
        "pass" if supervised[target_cols].isna().sum().sum() == 0 else "fail",
        "pass" if split_summary["start"].is_monotonic_increasing and split_summary["end"].is_monotonic_increasing else "review",
    ],
})


## 13. Assemble and Write Preprocessed Artifacts


In [ ]:
id_cols = ["timestamp", "facility_id", "split"]
audit_context_cols = [
    "breach_now_bool", "sensor_status", "was_inserted_timestamp",
    "has_full_history_window", "row_number_by_facility",
]
optional_context = ["compliance_status", "main_risk_driver", "current_breach_reason", "event_type"]
context_cols = [col for col in audit_context_cols + optional_context if col in supervised.columns]

model_matrix = pd.concat(
    [
        supervised[id_cols + context_cols + target_cols + forecast_target_cols].reset_index(drop=True),
        imputed_df[kept_imputed_features].reset_index(drop=True),
    ],
    axis=1,
)
for col in target_cols:
    model_matrix[col] = model_matrix[col].astype(int)

split_summary.to_csv(OUTPUTS["split_summary"], index=False)
manifest.to_csv(OUTPUTS["feature_manifest"], index=False)
train_medians.to_csv(OUTPUTS["train_medians"], index=False)
model_matrix.to_csv(OUTPUTS["model_matrix"], index=False)

audit = {
    "input_rows": int(len(operational)),
    "regularised_rows": int(len(work)),
    "supervised_rows": int(len(supervised)),
    "duplicate_rows_removed": int(duplicate_rows_removed),
    "inserted_timestamp_rows": int(inserted_timestamp_rows),
    "sample_interval_minutes": int(sample_minutes),
    "max_lag_minutes": int(max_lag_steps * sample_minutes),
    "max_prediction_horizon_minutes": int(max_horizon),
    "feature_policy": "real-time sensor and soft-sensor features only; lab/post-hoc/compliance explanation columns excluded",
    "imputation_policy": "short gaps forward-filled causally before feature generation; remaining values median-imputed using train split only",
    "split_policy": "chronological 70/15/15 after warm-up and known target filtering",
    "raw_feature_count": int(len(feature_cols)),
    "candidate_feature_count": int(len(candidate_feature_cols)),
    "kept_feature_count": int(len(kept_imputed_features)),
    "all_nan_feature_count": int(len(all_nan_features)),
    "low_variance_feature_count": int(len(low_variance_features)),
    "high_correlation_drop_count": int(len(high_corr_drop)),
    "target_columns": target_cols,
    "forecast_target_columns": forecast_target_cols,
    "excluded_predictor_columns": post_hoc_or_unavailable,
    "outputs": {name: str(path.relative_to(ROOT)) for name, path in OUTPUTS.items()},
}
OUTPUTS["audit"].write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")

pd.DataFrame({
    "artifact": OUTPUTS.keys(),
    "path": [str(path.relative_to(ROOT)) for path in OUTPUTS.values()],
    "exists": [path.exists() for path in OUTPUTS.values()],
})


## 14. Final Quality Gates


In [ ]:
assert model_matrix["timestamp"].is_monotonic_increasing, "Model matrix is not globally timestamp-sorted"
assert not model_matrix[kept_imputed_features].isna().any().any(), "Kept feature matrix still contains NaNs"
assert model_matrix[target_cols].isin([0, 1]).all().all(), "Classification targets must be binary after filtering"
assert set(model_matrix["split"].unique()) == {"train", "validation", "test"}, "Expected train/validation/test splits"
assert not feature_leakage_hits, "Leakage-like features found"

final_summary = {
    "model_matrix_shape": model_matrix.shape,
    "train_validation_test_rows": model_matrix["split"].value_counts().to_dict(),
    "target_positive_rates_pct": model_matrix[target_cols].mean().mul(100).round(3).to_dict(),
    "kept_feature_count": len(kept_imputed_features),
    "first_timestamp": str(model_matrix["timestamp"].min()),
    "last_timestamp": str(model_matrix["timestamp"].max()),
}
final_summary


## 15. Handoff Notes for the Training Notebook

Use `data/processed/preprocessed_model_matrix.csv` for model training.

Recommended first target:

```text
target_breach_next_30min
```

Training guardrails:

- never random-split this matrix;
- train only on rows where `split == "train"`;
- tune thresholds only on `split == "validation"`;
- report final metrics once on `split == "test"`;
- keep audit/context columns out of `X` and use only columns ending in `__imputed` where `keep_for_model == True` in the manifest.
